# Notebook 2: Portfolio Summary (Easy - SQL + Python)
Combine SQL queries with Python pandas for portfolio breakdown and formatted output.

In [ ]:
%%sql -r loan_details
SELECT l.LOAN_ID, l.LOAN_TYPE, l.LOAN_AMOUNT, l.INTEREST_RATE, l.TERM_MONTHS,
       l.LOAN_STATUS, l.PURPOSE, l.MONTHLY_PAYMENT,
       c.FIRST_NAME || ' ' || c.LAST_NAME AS CUSTOMER_NAME,
       c.CREDIT_SCORE, c.ANNUAL_INCOME, c.STATE
FROM LOAN_DB.LENDING.LOANS l
JOIN LOAN_DB.LENDING.CUSTOMERS c ON l.CUSTOMER_ID = c.CUSTOMER_ID
ORDER BY l.LOAN_AMOUNT DESC

In [ ]:
import pandas as pd

df = loan_details.copy()
print(f"Total Loans: {len(df)}")
print(f"Total Portfolio: ${df['LOAN_AMOUNT'].sum():,.2f}")
print(f"Average Loan: ${df['LOAN_AMOUNT'].mean():,.2f}")
print(f"\nLoan Types: {df['LOAN_TYPE'].nunique()}")
print(f"States Covered: {df['STATE'].nunique()}")

In [ ]:
summary = df.groupby('LOAN_TYPE').agg(
    count=('LOAN_ID', 'count'),
    total_amount=('LOAN_AMOUNT', 'sum'),
    avg_rate=('INTEREST_RATE', 'mean'),
    avg_credit_score=('CREDIT_SCORE', 'mean')
).round(2)

summary['pct_of_portfolio'] = (summary['total_amount'] / summary['total_amount'].sum() * 100).round(1)
summary = summary.sort_values('total_amount', ascending=False)
summary

In [ ]:
status_pivot = df.pivot_table(
    index='LOAN_TYPE',
    columns='LOAN_STATUS',
    values='LOAN_AMOUNT',
    aggfunc='sum',
    fill_value=0
)
status_pivot['TOTAL'] = status_pivot.sum(axis=1)
status_pivot = status_pivot.sort_values('TOTAL', ascending=False)
status_pivot

In [ ]:
bins = [0, 10000, 50000, 100000, 250000, 700000]
labels = ['<10K', '10K-50K', '50K-100K', '100K-250K', '250K+']
df['SIZE_BUCKET'] = pd.cut(df['LOAN_AMOUNT'], bins=bins, labels=labels)

bucket_summary = df.groupby('SIZE_BUCKET', observed=True).agg(
    count=('LOAN_ID', 'count'),
    total=('LOAN_AMOUNT', 'sum'),
    avg_rate=('INTEREST_RATE', 'mean')
).round(2)
bucket_summary